In [1]:
import cv2
import numpy as np



In [2]:
# CAM_INDEX: indice de la camara activa
CAM_INDEX = 1
ROI_W = 0.55
ROI_H = 0.45
MIN_AREA_FLOOR = 700
MIN_AREA_RATIO = 0.006
BORDER_MARGIN = 1
BORDER_REJECT_AREA_RATIO = 0.98
# Sensibilidad de bordes internos: umbral bajo de Canny (mas bajo = mas detalle y mas ruido)
INTERNAL_EDGE_CANNY_LOW = 17
# Sensibilidad de bordes internos: umbral alto de Canny (mas alto = menos ruido, menos bordes)
INTERNAL_EDGE_CANNY_HIGH = 45
# Iteraciones de dilatacion para engrosar bordes internos antes de medir su densidad
INTERNAL_EDGE_DILATE_ITERS = 2
# Densidad minima de borde interno para activar reglas de detalle interno (ej. bicolor)
INTERNAL_EDGE_DENSITY_THRESHOLD = 0.019

print('q = salir')


SHAPE_NAMES = ['desconocida', 'circular', 'capleta', 'pastilla']
SHAPE_UNKNOWN = 0
SHAPE_CIRCULAR = 1
SHAPE_CAPLETA = 2
SHAPE_PASTILLA = 3



q = salir


In [3]:
# color -> rangos HSV (h_min, h_max, s_min, s_max, v_min, v_max)
# COLOR_ORDER define prioridad: el primer color que coincide se asigna.
# Si un tono cae en varios rangos, gana el color que aparezca antes.
# Esto ayuda a resolver conflictos como naranja vs cafe o rosa vs rojo.
COLOR_ORDER = ['negro', 'azul', 'rosa', 'cafe', 'naranja', 'amarillo', 'rojo']
COLOR_RANGOS = {
    'negro':    [(0, 179, 0, 15, 0, 39), (0, 179, 0, 21, 0, 94)],
    'azul':     [(82, 132, 14, 255, 70, 255)],
    'rosa':     [(136, 175, 20, 255, 85, 255), (165, 179, 0, 130, 95, 255)],
    'cafe':     [(8, 20, 140, 255, 0, 130), (8, 24, 110, 255, 0, 110)],
    'naranja':  [(7, 22, 45, 255, 125, 255)],
    'amarillo': [(24, 36, 150, 255, 105, 255), (22, 38, 120, 255, 105, 255)],
    'rojo':     [(0, 5, 40, 255, 60, 255), (176, 179, 40, 255, 60, 255)],
}

def classify_hsv_pixels(px):
    if len(px) == 0:
        return np.empty((0,), dtype=object)
    h = px[:, 0].astype(np.float32)
    s = px[:, 1].astype(np.float32)
    v = px[:, 2].astype(np.float32)
    labels = np.full(h.shape, 'negro', dtype=object)
    assigned = np.zeros(h.shape, dtype=bool)
    for cname in COLOR_ORDER:
        for h_min, h_max, s_min, s_max, v_min, v_max in COLOR_RANGOS[cname]:
            m = (~assigned) & (h >= h_min) & (h <= h_max) & (s >= s_min) & (s <= s_max) & (v >= v_min) & (v <= v_max)
            if np.any(m):
                labels[m] = cname
                assigned[m] = True
    m_yellow = (~assigned) & (h >= 39) & (h <= 81)
    if np.any(m_yellow):
        labels[m_yellow] = 'amarillo'
        assigned[m_yellow] = True
    m_orange = (~assigned) & (h >= 6) & (h <= 21)
    if np.any(m_orange):
        labels[m_orange & (v >= 145)] = 'naranja'
        labels[m_orange & (v < 145)] = 'cafe'
    return labels


def classify_hsv_scalar(hh, ss, vv):
    for cname in COLOR_ORDER:
        for h_min, h_max, s_min, s_max, v_min, v_max in COLOR_RANGOS[cname]:
            if h_min <= hh <= h_max and s_min <= ss <= s_max and v_min <= vv <= v_max:
                return cname
    if 39 <= hh <= 81:
        return 'amarillo'
    if 6 <= hh <= 21:
        return 'naranja' if vv >= 145 else 'cafe'
    return 'negro'


# fallback de color (en el loop):
# si no cae en un rango HSV, se fuerza a amarillo o naranja/cafe por tono.



In [4]:
# inicializa camara principal
cap = cv2.VideoCapture(CAM_INDEX)
if not cap.isOpened():
    # fallback para equipos Windows donde DSHOW abre mejor la camara
    cap = cv2.VideoCapture(CAM_INDEX, cv2.CAP_DSHOW)

if not cap.isOpened():
    raise RuntimeError(f'No se pudo abrir la camara {CAM_INDEX}')

KERNEL_OPEN = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
KERNEL_CLOSE = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9))
KERNEL_ERODE_SEP = np.ones((3, 3), np.uint8)
KERNEL_ERODE_INNER = np.ones((5, 5), np.uint8)
KERNEL_DILATE_INNER = np.ones((3, 3), np.uint8)

def infer_brand(shape_label, primary_color, secondary_color, is_bicolor):
    colors = {primary_color}
    if is_bicolor and secondary_color:
        colors.add(secondary_color)

    # Reglas bicolor
    if {'naranja', 'amarillo'}.issubset(colors) and shape_label == 'pastilla':
        return 'Teva Pharmaceuticals USA'
    if {'azul', 'naranja'}.issubset(colors) and shape_label == 'pastilla':
        return 'Alembic Pharmaceuticals'
    if {'negro', 'amarillo'}.issubset(colors):
        return 'Zydus Pharmaceuticals'

    # Reglas por forma y color
    if shape_label == 'circular' and primary_color == 'rojo':
        return 'Glaxo SmithKline'
    if shape_label == 'circular' and primary_color == 'azul':
        return 'Glaxo SmithKline'
    if shape_label == 'capleta' and primary_color == 'azul':
        return 'Glaxo SmithKline'
    if shape_label == 'capleta' and primary_color == 'cafe':
        return 'Heritage Pharmaceuticals'
    if shape_label == 'circular' and primary_color in {'cafe', 'rojo', 'naranja'}:
        return 'Cipla USA Inc.'

    return ''

while True:
    # toma frame actual de camara
    ok, frame = cap.read()
    if not ok:
        continue

    display = frame
    h0, w0 = frame.shape[:2]

    rw = int(w0 * ROI_W)
    rh = int(h0 * ROI_H)
    x1 = (w0 - rw) // 2
    y1 = (h0 - rh) // 2
    x2 = x1 + rw
    y2 = y1 + rh
    roi = frame[y1:y2, x1:x2]

    cv2.rectangle(display, (x1, y1), (x2, y2), (0, 255, 255), 2)
    cv2.putText(display, 'ROI', (x1, max(25, y1 - 10)), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)

    # estima color del papel usando 4 esquinas de la ROI
    h, w = roi.shape[:2]
    ph = max(12, int(h * 0.10))
    pw = max(12, int(w * 0.10))
    patches = [
        roi[0:ph, 0:pw],
        roi[0:ph, w-pw:w],
        roi[h-ph:h, 0:pw],
        roi[h-ph:h, w-pw:w],
    ]
    paper_bgr = np.median(np.concatenate([p.reshape(-1, 3) for p in patches], axis=0), axis=0).astype(np.uint8)

    # segmenta por distancia al color de papel (LAB)
    lab = cv2.cvtColor(roi, cv2.COLOR_BGR2LAB).astype(np.float32)
    paper_lab = cv2.cvtColor(np.uint8([[paper_bgr]]), cv2.COLOR_BGR2LAB)[0, 0].astype(np.float32)
    delta = np.linalg.norm(lab - paper_lab, axis=2)
    delta = cv2.normalize(delta, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    delta = cv2.GaussianBlur(delta, (5, 5), 0)

    otsu_thr, _ = cv2.threshold(delta, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    mask = (delta > max(8, int(otsu_thr) - 10)).astype(np.uint8) * 255

    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, KERNEL_OPEN, iterations=1)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, KERNEL_CLOSE, iterations=1)

    # separa objetos cercanos para evitar fusion cuando la pastilla esta lejos
    mask_sep = cv2.erode(mask, KERNEL_ERODE_SEP, iterations=1)

    contours_all, _ = cv2.findContours(mask_sep, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    roi_area = float(h * w)
    min_area = max(MIN_AREA_FLOOR, int(roi_area * MIN_AREA_RATIO))

    contours = []
    # filtra contornos invalidos (ruido, tiras, blobs extremos)
    for cnt in contours_all:
        area = cv2.contourArea(cnt)
        if area < min_area:
            continue

        x, y, cw, ch = cv2.boundingRect(cnt)
        box_area = max(float(cw * ch), 1.0)
        fill_ratio = area / box_area
        aspect_box = max(cw, ch) / max(min(cw, ch), 1)

        if fill_ratio < 0.28:
            continue
        if aspect_box > 6.0:
            continue

        touches_border = (
            x <= BORDER_MARGIN
            or y <= BORDER_MARGIN
            or (x + cw) >= (w - BORDER_MARGIN)
            or (y + ch) >= (h - BORDER_MARGIN)
        )
        if touches_border and area > roi_area * BORDER_REJECT_AREA_RATIO and len(contours_all) > 1:
            continue

        contours.append(cnt)

    contours = sorted(contours, key=cv2.contourArea, reverse=True)
    primary = None

    # si hay dos mitades alineadas, las une como una sola pastilla
    if len(contours) >= 2:
        c1, c2 = contours[0], contours[1]
        a1, a2 = cv2.contourArea(c1), cv2.contourArea(c2)
        if a1 > 1 and (a2 / a1) >= 0.45:
            x1b, y1b, w1b, h1b = cv2.boundingRect(c1)
            x2b, y2b, w2b, h2b = cv2.boundingRect(c2)
            c1x, c1y = x1b + w1b / 2.0, y1b + h1b / 2.0
            c2x, c2y = x2b + w2b / 2.0, y2b + h2b / 2.0
            y_aligned = abs(c1y - c2y) <= 0.35 * max(h1b, h2b)
            x_separated = abs(c1x - c2x) >= 0.25 * max(w1b, w2b)
            if y_aligned and x_separated:
                primary = cv2.convexHull(np.vstack([c1, c2]))

    if primary is None and len(contours) > 0:
        cx0, cy0 = w / 2.0, h / 2.0
        best_score = -1e9
        for cnt in contours[:5]:
            area = cv2.contourArea(cnt)
            x, y, cw, ch = cv2.boundingRect(cnt)
            cx = x + cw / 2.0
            cy = y + ch / 2.0
            center_dist = np.hypot(cx - cx0, cy - cy0) / max(np.hypot(cx0, cy0), 1e-6)
            area_norm = area / max(float(w * h), 1.0)
            # penaliza blobs demasiado grandes (suelen ser fusion con figura vecina)
            size_penalty = max(0.0, (cw / max(w, 1)) - 0.55) * 1.5 + max(0.0, (ch / max(h, 1)) - 0.55) * 1.5
            score = (2.0 * area_norm) - (0.75 * center_dist) - size_penalty
            if score > best_score:
                best_score = score
                primary = cnt

    roi_vis = roi.copy()
    edge_ext_vis = np.zeros(roi.shape[:2], dtype=np.uint8)
    edge_int_vis = np.zeros(roi.shape[:2], dtype=np.uint8)

    if primary is None:
        # sin candidato util despues de segmentar y filtrar
        cv2.putText(roi_vis, 'Sin pastilla detectada', (15, 28), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
    else:
        hull = cv2.convexHull(primary)
        peri_h = cv2.arcLength(hull, True)
        eps = max(1.0, 0.008 * peri_h)
        approx = cv2.approxPolyDP(hull, eps, True)
        cnt = approx if len(approx) >= 5 else hull

        # clasifica forma con reglas geometricas simples
        area = cv2.contourArea(cnt)
        peri = cv2.arcLength(cnt, True)
        circularity = 0.0 if peri == 0 else (4 * np.pi * area / (peri * peri))
        (_, _), (rw2, rh2), _ = cv2.minAreaRect(cnt)
        rw2 = max(rw2, 1e-6)
        rh2 = max(rh2, 1e-6)
        aspect = max(rw2, rh2) / min(rw2, rh2)

        shape_idx = SHAPE_UNKNOWN
        if circularity >= 0.84 and aspect <= 1.12:
            shape_idx = SHAPE_CIRCULAR
        elif 1.20 <= aspect < 3.60 and circularity >= 0.50:
            shape_idx = SHAPE_CAPLETA
        elif aspect >= 3.60 or (aspect >= 2.30 and circularity < 0.50):
            shape_idx = SHAPE_PASTILLA
        elif aspect > 1.12:
            shape_idx = SHAPE_CAPLETA

        if shape_idx == SHAPE_PASTILLA and aspect < 2.45 and circularity >= 0.62:
            shape_idx = SHAPE_CAPLETA
        if shape_idx == SHAPE_CIRCULAR and aspect > 1.20:
            shape_idx = SHAPE_CAPLETA

        shape_label = SHAPE_NAMES[shape_idx]

        cv2.drawContours(edge_ext_vis, [cnt], -1, 255, 2)

        obj = np.zeros(roi.shape[:2], dtype=np.uint8)
        cv2.drawContours(obj, [cnt], -1, 255, -1)
        inner = cv2.erode(obj, KERNEL_ERODE_INNER, iterations=1)
        gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
        gray = cv2.GaussianBlur(gray, (5, 5), 0)
        edges = cv2.Canny(gray, INTERNAL_EDGE_CANNY_LOW, INTERNAL_EDGE_CANNY_HIGH)
        edge_int_vis = cv2.bitwise_and(edges, inner)
        edge_int_vis = cv2.dilate(edge_int_vis, KERNEL_DILATE_INNER, iterations=INTERNAL_EDGE_DILATE_ITERS)
        pix_in = max(cv2.countNonZero(inner), 1)
        int_edge_density = float(cv2.countNonZero(edge_int_vis) / pix_in)

        roi_vis[edge_ext_vis > 0] = (0, 255, 0)
        roi_vis[edge_int_vis > 0] = (0, 0, 255)

        # clasifica color principal dentro del contorno
        hsv = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)
        obj_mask = np.zeros(roi.shape[:2], dtype=np.uint8)
        cv2.drawContours(obj_mask, [cnt], -1, 255, -1)
        inner_col = cv2.erode(obj_mask, KERNEL_ERODE_INNER, iterations=1)
        if cv2.countNonZero(inner_col) < 30:
            inner_col = obj_mask

        pixels = hsv[inner_col == 255]
        if len(pixels) > 0:
            px = pixels[pixels[:, 2] < 245]
            if len(px) == 0:
                px = pixels
            h_med = float(np.median(px[:, 0]))
            s_med = float(np.percentile(px[:, 1], 40))
            v_med = float(np.percentile(px[:, 2], 75))
        else:
            px = np.empty((0, 3), dtype=np.uint8)
            h_med, s_med, v_med = 0.0, 0.0, 0.0

        labels = classify_hsv_pixels(px)
        if len(labels) > 0:
            uniq, cnts = np.unique(labels, return_counts=True)
            counts = {k: int(v) for k, v in zip(uniq.tolist(), cnts.tolist())}
        else:
            counts = {}
        total = max(int(len(labels)), 1)
        filtered = {k: v for k, v in counts.items() if (v / total) >= 0.03}
        if not filtered:
            filtered = counts if counts else {'negro': 1}
        ranked = sorted(filtered.items(), key=lambda kv: kv[1], reverse=True)
        primary_color = ranked[0][0]
        primary_ratio = ranked[0][1] / total

        # color central robusto para estabilizar mono-color
        hh = h_med; ss = s_med; vv = v_med
        central_color = classify_hsv_scalar(hh, ss, vv)

        if primary_ratio < 0.72 or primary_color == 'unknown':
            primary_color = central_color

        # detecta bicolor separando el objeto en 2 mitades
        secondary_color = ''
        is_bicolor = False
        if shape_label in ('pastilla', 'capleta') and (aspect >= 1.45 or int_edge_density >= INTERNAL_EDGE_DENSITY_THRESHOLD):
            ys, xs = np.where(inner_col == 255)
            if len(xs) >= 120:
                bx, by, bw, bh = cv2.boundingRect(cnt)
                if bw >= bh:
                    split_v = np.median(xs)
                    left_sel = xs < split_v
                    right_sel = xs >= split_v
                else:
                    split_v = np.median(ys)
                    left_sel = ys < split_v
                    right_sel = ys >= split_v

                if np.count_nonzero(left_sel) >= 40 and np.count_nonzero(right_sel) >= 40:
                    side_names = []
                    side_ratios = []
                    for sel in [left_sel, right_sel]:
                        side_pixels = hsv[ys[sel], xs[sel]]
                        side_pixels = side_pixels[side_pixels[:, 2] < 245]
                        if len(side_pixels) == 0:
                            side_names.append('unknown')
                            side_ratios.append(0.0)
                        else:
                            side_labels = classify_hsv_pixels(side_pixels)
                            if len(side_labels) > 0:
                                su, sc = np.unique(side_labels, return_counts=True)
                                scount = {k: int(v) for k, v in zip(su.tolist(), sc.tolist())}
                            else:
                                scount = {}
                            if not scount:
                                side_names.append('unknown')
                                side_ratios.append(0.0)
                            else:
                                k, v = max(scount.items(), key=lambda kv: kv[1])
                                side_names.append(k)
                                side_ratios.append(float(v / max(int(len(side_labels)), 1)))

                    if side_names[0] != 'unknown' and side_names[1] != 'unknown' and side_names[0] != side_names[1] and side_ratios[0] >= 0.30 and side_ratios[1] >= 0.30:
                        is_bicolor = True
                        primary_color = side_names[0]
                        secondary_color = side_names[1]
                        shape_label = 'pastilla'

        if shape_label != 'pastilla':
            is_bicolor = False
            secondary_color = ''

        x, y, bw, bh = cv2.boundingRect(cnt)
        cv2.drawContours(roi_vis, [cnt], -1, (0, 255, 0), 2)
        cv2.rectangle(roi_vis, (x, y), (x + bw, y + bh), (255, 0, 0), 2)

        ratio_size = area / max(float(h * w), 1.0)
        if ratio_size >= 0.20:
            size_label = 'grande'
        elif ratio_size >= 0.10:
            size_label = 'mediano'
        else:
            size_label = 'pequeno'

        color_label = f"{primary_color}/{secondary_color}" if is_bicolor else primary_color
        brand_label = infer_brand(shape_label, primary_color, secondary_color, is_bicolor)
        label = f"{color_label} | {shape_label} | {size_label}"
        cv2.putText(roi_vis, label, (x, max(20, y - 8)), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 255, 0), 2)

        if brand_label:
            cv2.putText(roi_vis, f"Marca: {brand_label}", (x, y + bh + 72), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 255, 255), 1)

        hsv_txt = f"H:{h_med:.0f} S:{s_med:.0f} V:{v_med:.0f}"
        cv2.putText(roi_vis, hsv_txt, (x, y + bh + 18), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 255, 255), 1)
        edge_txt = f"Eint:{int_edge_density:.3f}"
        cv2.putText(roi_vis, edge_txt, (x, y + bh + 36), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 255, 255), 1)
        size_txt = f"Tam:{size_label} A:{ratio_size:.3f}"
        cv2.putText(roi_vis, size_txt, (x, y + bh + 54), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 255, 255), 1)

    paper_box = np.full((40, 120, 3), paper_bgr, dtype=np.uint8)
    # paper_ref: color de referencia del papel estimado por esquinas
    #cv2.imshow('paper_ref', paper_box)
    # camera: frame completo con ROI dibujada
    cv2.imshow('camera', display)
    # delta_white: distancia al color del papel (base de segmentación)
    #cv2.imshow('delta_white', delta)
    # mask: segmentación binaria principal del objeto
    #cv2.imshow('mask', mask)
    # mask_sep: versión erosionada para separar objetos cercanos
    #cv2.imshow('mask_sep', mask_sep)
    # edge_ext: borde externo del contorno elegido
    #cv2.imshow('edge_ext', edge_ext_vis)
    # edge_int: bordes internos (útil para bicolor y detalle interno)
    #cv2.imshow('edge_int', edge_int_vis)
    # roi_result: overlay final con color, forma, tamaño y métricas
    cv2.imshow('roi_result', roi_vis)

    # tecla de salida
    key = cv2.waitKey(1) & 0xFF
    if key == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


RuntimeError: No se pudo abrir la camara 1